# Composio Python SDK

[Composio](https://composio.dev) is a platform for connecting AI agents to external tools and
services (GitHub, Slack, Jira, Google Workspace, and hundreds more). This notebook shows how
to use its Python client to explore available toolkits and tools directly from a Cribl notebook.

## What you'll learn

1. Install and initialise the Composio HTTP client in Pyodide
2. List all toolkits (integrations) available on the Composio platform
3. Explore the tools inside a specific toolkit
4. Use an AI prompt to generate a follow-up analysis

## Setup

Run the cell below **once**. It installs `composio-client` — the official Stainless-generated
OpenAPI client that the Composio Python SDK uses internally.

> **Why `composio-client` and not `composio`?**  
> The top-level `composio` package requires `openai`, which depends on `jiter` (a Rust
> extension). No Pyodide/WASM wheel exists for `jiter`, so `micropip` cannot install it.
> `composio-client` is the underlying HTTP layer that does all the actual API calls — it is
> entirely pure Python and installs cleanly in Pyodide.

> **Why install `attrs` first?**  
> Pyodide's auto-import hook maps an unknown `import attr` to `micropip.install('attr')`,
> which installs a *different*, unrelated PyPI package named `attr`. Installing `attrs`
> (the correct package) first ensures the right module is already in `sys.modules`.

In [ ]:
import micropip
# 'attrs' must be installed before composio-client.
# Pyodide's auto-import hook uses the import name 'attr' to look up a PyPI package,
# but the PyPI package named 'attr' is unrelated to 'attrs' and lacks the .s decorator.
# Pre-installing 'attrs' ensures the correct module is in sys.modules first.
await micropip.install('attrs')
await micropip.install('composio-client==1.39.0')

## Credentials (one-time setup)

Run the cell below **once** to save your Composio API key into the app's secure key store.  
After that the platform proxy injects `x-api-key` automatically on every request — you never
need to paste the key again.

> **Why KV store instead of a plain variable?**  
> The Pyodide kernel bridges `fetch()` calls from the worker thread to the main thread and
> deliberately strips all custom request headers so the Cribl platform can inject its own auth.
> That stripping also removes any `x-api-key` set by `composio-client`. Saving the key to the
> app's KV store lets the proxy inject it at the network layer, bypassing the strip.

**Where to get your API key:**
1. Log in at [app.composio.dev](https://app.composio.dev)
2. Click your avatar → **Settings** → **API Keys**
3. Copy an existing key or click **Create new API key**

In [ ]:
import os
from pyodide.http import pyfetch

# Paste your real Composio API key here, then run this cell.
# The key is saved to the app's KV store; after this you never need to re-enter it.
COMPOSIO_API_KEY = "<YOUR_COMPOSIO_API_KEY>"

if COMPOSIO_API_KEY.startswith("<"):
    raise RuntimeError(
        "Replace the COMPOSIO_API_KEY placeholder above with your real key, then re-run this cell."
    )

_cribl_api = os.getenv("CRIBL_API_URL", "") or str(getattr(__import__("js"), "CRIBL_API_URL", ""))
if not _cribl_api:
    raise RuntimeError("CRIBL_API_URL is not set. Are you running inside Cribl?")

resp = await pyfetch(
    f"{_cribl_api}/kvstore/composioApiKey",
    method="PUT",
    body=COMPOSIO_API_KEY,
)
if resp.status not in (200, 201, 204):
    raise RuntimeError(f"Failed to save API key to KV store (HTTP {resp.status}).")

print("API key saved. The proxy will inject x-api-key on every Composio request from now on.")

## Initialise the client

Create a `Composio` client with no explicit API key — the proxy injects `x-api-key`
from the KV store you populated above, so the SDK itself does not need the value.

In [ ]:
from composio_client import Composio

# No api_key argument — the platform proxy injects x-api-key from kv.composioApiKey.
client = Composio()
print("Composio client ready.")

## Checkpoint 1 — List available toolkits

Toolkits are named integrations (e.g. `github`, `slack`, `jira`). This call returns
everything available on the platform — no connected account required.

In [ ]:
toolkits_page = client.toolkits.list()

# .items holds the list of toolkit objects
toolkits = list(toolkits_page.items or [])
print(f"Total toolkits: {len(toolkits)}")

# Show the first 15 slugs
for tk in toolkits[:15]:
    print(f"  {tk.slug}")

## Checkpoint 2 — Explore tools inside a toolkit

Each toolkit exposes a set of tools (API actions). Change `TOOLKIT_SLUG` to any slug
from the list above.

In [ ]:
TOOLKIT_SLUG = "github"  # change to any slug from Checkpoint 1

tools_page = client.tools.list(toolkit_slug=TOOLKIT_SLUG)
tools = list(tools_page.items or [])

print(f"Tools in '{TOOLKIT_SLUG}': {len(tools)}")
for tool in tools[:10]:
    desc = getattr(tool, 'description', '') or ''
    print(f"  {tool.slug:<45} {desc[:60]}")

## Checkpoint 3 — Summarise toolkit coverage

Build a quick bar chart showing the tool count across a sample of toolkits.

In [ ]:
import matplotlib.pyplot as plt

# Sample up to 12 toolkits for the chart
sample_slugs = [tk.slug for tk in toolkits[:12]]
counts = []
for slug in sample_slugs:
    page = client.tools.list(toolkit_slug=slug)
    counts.append(len(list(page.items or [])))

plt.figure(figsize=(10, 4))
plt.bar(sample_slugs, counts)
plt.xticks(rotation=40, ha='right')
plt.ylabel('Number of tools')
plt.title('Tool count by toolkit (first 12)')
plt.tight_layout()
plt.show()

## Next steps

- **Connect a service:** visit [app.composio.dev/connections](https://app.composio.dev/connections)
  to link a GitHub, Slack, or other account, then call `client.tools.execute(tool_slug=..., user_id=...)`
  to run actions on behalf of that user.
- **Browse all toolkits:** [app.composio.dev/tools](https://app.composio.dev/tools)
- **Open `AI_Magic.ipynb`** to practice iterative prompt refinement on the data you fetched here.

In [ ]:
# ### Prompt:
# Analyse the Composio toolkit and tool data fetched above.
# Context:
# - toolkits: list of toolkit objects, each with .slug
#   {{ toolkits | describe }}
# - tools: tools in the '{{ TOOLKIT_SLUG }}' toolkit
#   {{ tools | describe }}
#
# Requirements:
# - Summarise the most interesting or useful tools in 3-5 bullet points
# - Suggest 2-3 automation workflows that could be built with these tools
# - Keep the response concise